# Gold Work Incremental
## Step 1 : Imports and Setups

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog novacart_adb")
spark.sql("create schema if not exists gold")
gold_run_id = uuid.uuid4()
run_ts_str = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
run_date_str = datetime.utcnow().strftime("%Y-%m-%d")

print("Current Gold Run Id :",gold_run_id)
print("Run Timestamp Folder :",run_ts_str)

## Step 2 : Gold Control Table
 This table stores the the latest Gold processing state.

In [0]:
spark.sql("""
          Create table if not exists novacart_adb.gold.processing_control(
              layer string,
              entity_name string,
              last_processed_silver_run_id  string,
              last_processed_silver_run_ts timestamp,
              rows_merged bigint,
              run_status string,
              gold_run_id string,
              updated_at timestamp
          )
          using DELTA
          """)

# Step 3 : Helper Functions
This cell defines reusable Gold functions

- upsert_to_gold() merge data into Gold current - state tables
- get_last_processed_silver_ts() reads the Gold watermark from the control table
 - upsert_gold_control() updates gold control after a run sucessful


In [0]:
def upsert_to_gold(df_source,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (dt.alias("t")
         .merge(df_source.alias("s"),f"t.{join_key} = s.{join_key}"
        ).whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_silver_ts(entity_name:str):
    ctrl = (spark.table("gold.processing_control")
        .filter(
            (F.col("layer") == "silver")
            & (F.col("entity_name") == entity_name)
            & (F.col("run_status") == "success")
        )
        .orderBy(F.col("updated_at").desc()).limit(1)
    )

    rows = ctrl.collect()
    if not rows:
        return None
    return rows[0]['last_processed_silver_run_ts']

In [0]:
def upsert_gold_control(entity_name,last_processed_silver_run_id,last_processed_silver_run_ts,rows_merged):
    ctrl_df = spark.createDataFrame(
        [("silver",
          entity_name,
          last_processed_silver_run_id,
          last_processed_silver_run_ts,
          int(rows_merged),
          "success",
          str(gold_run_id),
          datetime.now()
          )],
        schema = """
        layer string,
        entity_name string,
        last_processed_silver_run_id string,
        last_processed_silver_run_ts timestamp,
        rows_merged bigint,
        run_status string,
        gold_run_id string,
        updated_at timestamp
        """
    )
    dt = DeltaTable.forName(spark, "gold.processing_control")
    (dt.alias("t")
         .merge(ctrl_df.alias("s"),f"t.layer = s.layer AND t.entity_name = s.entity_name"
        ).whenMatchedUpdate(set={
            "last_processed_silver_run_id":"s.last_processed_silver_run_id",
            "last_processed_silver_run_ts":"s.last_processed_silver_run_ts",
            "rows_merged":"s.rows_merged",
            "run_status":"s.run_status",
            "gold_run_id":"s.gold_run_id",
            "updated_at":"s.updated_at"
            })
         .whenNotMatchedInsertAll()
         .execute())
   

## Step 4 : Read Changed Silver Rows only
This cell reads the full Silver cuurent state tables but filters out only the rows that changed since the the last run

This is the starting point for Gold Incremental Processing

In [0]:
last_gold_ts = get_last_processed_silver_ts("orders_information")
print("Last Processed Silver Timestamp for Gold = ",last_gold_ts)

silver_orders_current = spark.read.table('novacart_adb.silver.orders_transformed')
silver_products_current = spark.read.table('novacart_adb.silver.products_transformed')
silver_payments_current = spark.read.table('novacart_adb.silver.payments_transformed')

if last_gold_ts is None:
    changed_orders = silver_orders_current
    changed_products = silver_products_current
    changed_payments = silver_payments_current
else:
    changed_orders = silver_orders_current.filter(F.col("updated_at") >F.lit(last_gold_ts))
    changed_products = silver_products_current.filter(F.col("updated_at") >F.lit(last_gold_ts))
    changed_payments = silver_payments_current.filter(F.col("updated_at") >F.lit(last_gold_ts))
    
changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print("Number of changed orders = ",changed_orders_count)
print("Number of changed products = ",changed_products_count)
print("Number of changed payments = ",changed_payments_count)

## Step 5 : Find impacted Order IDs

In [0]:
impacted_from_orders = changed_orders.select("order_id").distinct()
impacted_from_payments = changed_payments.select("order_id").distinct()
impacted_from_products = (
    changed_products.alias("p")
    .join (silver_orders_current.alias("o"), F.col("p.product_id") == F.col("o.product_id"),"inner")
    .select("o.order_id")
    .distinct()
)
impacted_order_id = (
    impacted_from_orders.union(impacted_from_payments)
    .union(impacted_from_products)
    .distinct()
)
print("impacted_order_id = ",impacted_order_id.count())
display(impacted_order_id.orderBy("order_id"))


## Step 6 : Bulid Gold Delta For Impacted orders

In [0]:
impacted_orders = (
    silver_orders_current
    .join(impacted_order_id, "order_id", "inner")
)
gold_delta =(
    impacted_orders.alias("o")
    .join(silver_products_current.alias("p"), F.col("o.product_id") == F.col("p.product_id"),"inner")
    .join(silver_payments_current.alias("pay"), F.col("o.order_id") == F.col("pay.order_id"),"inner")
    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("o.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("p.price").alias("product_price"),
        F.col("o.order_status"),
        F.col("o.order_amount"),
        F.col("pay.payment_id"),
        F.col("pay.payment_status"),
        F.col("pay.paid_amount"),
        F.col("o.order_date"),
        F.col("o.order_month"),
        F.col("o.order_year"),
        F.col("o.order_day"),
        F.greatest(
            F.col("o.updated_at").cast("timestamp"),
            F.col("p.updated_at").cast("timestamp"),
            F.col("pay.processed_at").cast("timestamp")
        ).alias("gold_update_ts")
    )
    .dropDuplicates(["order_id"])
    .withColumn("payment_completion_ratio",
                F.when(F.col("order_amount")>0,
                       F.col('paid_amount')/F.col('order_amount'))
                .otherwise(F.lit(0.0))
                )
    .withColumn(
        "payment_state",
        F.when(F.col('order_amount') == 0,'Invalid_order_amount')
         .when(F.col('payment_completion_ratio') == 0,'Unpaid')
         .when(F.col('payment_completion_ratio') == 1,'Paid')
         .when(F.col('payment_completion_ratio') < 1,'partially_paid')
         .when(F.col('payment_completion_ratio') > 1,'overpaid')
    )
    .withColumn("gold_update_date",F.to_date(F.col("gold_update_ts")))
    .withColumn("gold_run_id",F.lit(str(gold_run_id)))

)
display(gold_delta)

## Step 7 : Merge Gold Current-State Table

In [0]:
if gold_delta.count()>0:
    upsert_to_gold(gold_delta,'novacart_adb.gold.orders_information','order_id')
else:
    print("No new data to be inserted in gold table")

In [0]:
%sql
select * from novacart_adb.gold.orders_information

##Step 8 : Maintain Gold SCD Type2 History

In [0]:
if not spark.catalog.tableExists('novacart_adb.gold.orders_information_scd2'):
    spark.sql("""
        CREATE TABLE novacart_adb.gold.orders_information_scd2
        AS
        SELECT *, CAST(NULL AS TIMESTAMP) AS valid_from_ts, CAST(NULL AS TIMESTAMP) AS valid_to_ts, TRUE AS is_current 
        FROM novacart_adb.gold.orders_information
        WHERE 1 = 0
    """)

if gold_delta.count() > 0:
    gold_delta.createOrReplaceTempView("gold_delta_view")
    spark.sql("""
        MERGE INTO novacart_adb.gold.orders_information_scd2 t
        USING gold_delta_view s
        ON t.order_id = s.order_id AND t.is_current = true
        WHEN MATCHED AND (
            NOT (t.order_status <=> s.order_status) OR 
            NOT (t.order_amount <=> s.order_amount) OR 
            NOT (t.paid_amount <=> s.paid_amount) OR
            NOT (t.payment_id <=> s.payment_id) OR
            NOT (t.category <=> s.category) OR
            NOT (t.product_name <=> s.product_name) OR
            NOT (t.product_price <=> s.product_price)
        )
        THEN UPDATE SET
            t.valid_to_ts = s.gold_update_ts,
            t.is_current = false
    """)
    spark.sql("""
        INSERT INTO novacart_adb.gold.orders_information_scd2
        SELECT s.*, s.gold_update_ts AS valid_from_ts,
               CAST(NULL AS TIMESTAMP) AS valid_to_ts,
               TRUE AS is_current
        FROM gold_delta_view s
        LEFT JOIN novacart_adb.gold.orders_information_scd2 t
        ON s.order_id = t.order_id AND t.is_current = true
        WHERE t.order_id IS NULL OR (
            NOT (t.order_status <=> s.order_status) OR 
            NOT (t.order_amount <=> s.order_amount) OR 
            NOT (t.paid_amount <=> s.paid_amount) OR
            NOT (t.payment_id <=> s.payment_id) OR
            NOT (t.category <=> s.category) OR
            NOT (t.product_name <=> s.product_name) OR
            NOT (t.product_price <=> s.product_price)
        )
    """)

## Step 9: Update category-level Gold aggregation

In [0]:
if gold_delta.count()>0:
   impacted_categories = (gold_delta
   .select('category')
   .filter(F.col('category').isNotNull())
   .distinct()
   )
   category_performance_delta = (
       spark.read.table('novacart_adb.gold.orders_information')
       .join(impacted_categories, on='category', how='inner')
       .groupBy('category')
       .agg(F.countDistinct('order_id').alias('total_orders'),
           F.sum(
               F.when(F.col('order_amount')> 0,F.col("order_amount"))
               .otherwise(0.0)
           ).alias("Gross_merchandise_value"),
           F.sum(
               F.when(F.col('paid_amount')> 0,F.col("paid_amount"))
               .otherwise(0.0)
            ).alias("Total_paid_amount"),
           F.avg(F.col('payment_completion_ratio')).alias('avg_payment_completion_ratio'),
           (F.sum(F.when(F.col('payment_status')== 'FAILED',1).otherwise(0)) / F.count("*")).alias('payment_failure_rate')
           )
    )
   upsert_to_gold(category_performance_delta,'novacart_adb.gold.category_performance','category')
       
            
           
   

In [0]:
%sql
select * from novacart_adb.gold.category_performance

## Step 10 : Publish Gold snapshots to Volume

In [0]:
spark.sql("create volume if not exists novacart_adb.gold.gold_snapshots_volume")

In [0]:
latest_order_path = "/Volumes/novacart_adb/gold/gold_snapshots_volume/gold_latest/orders_information_scd2"
latest_category_path = "/Volumes/novacart_adb/gold/gold_snapshots_volume/gold_latest/category_performance"

histrocial_order_path = f"/Volumes/novacart_adb/gold/gold_snapshots_volume/gold_snapshots/orders_information/run_date = {run_date_str}/run_ts = {run_ts_str}"
historical_category_path = f"/Volumes/novacart_adb/gold/gold_snapshots_volume/gold_snapshots/category_performance/run_date = {run_date_str}/run_ts = {run_ts_str}"

spark.read.table("novacart_adb.gold.orders_information").write.mode("overwrite").format('parquet').save(latest_order_path)
spark.read.table("novacart_adb.gold.category_performance").write.mode("overwrite").format('parquet').save(latest_category_path)

spark.read.table("novacart_adb.gold.orders_information").write.mode("overwrite").format('parquet').save(histrocial_order_path)
spark.read.table("novacart_adb.gold.category_performance").write.mode("overwrite").format('parquet').save(historical_category_path)

# Grant access to the volume for a user or group
spark.sql("""
    GRANT READ VOLUME, WRITE VOLUME ON VOLUME novacart_adb.gold.gold_snapshots_volume TO `de.charan1610@gmail.com`
""")

print("latest order snapshot saved to :",latest_order_path)
print("latest category snapshot saved to :",latest_category_path)
print("historical order snapshot saved to :",histrocial_order_path)
print("historical category snapshot saved to :",historical_category_path)

## Update Gold control table

In [0]:
latest_silver_ts = silver_orders_current.agg(F.max('bronze_ingestion_at').alias('mx')).collect()[0]['mx']
latest_silver_run_id = (
    silver_orders_current
    .filter(F.col('bronze_ingestion_at')==latest_silver_ts)
    .agg(F.max('silver_run_id').alias('mx'))
    .collect()[0]['mx']
)if latest_silver_ts is not None else None

upsert_gold_control("orders_information",latest_silver_run_id,latest_silver_ts,gold_delta.count())
display(spark.sql('select * from novacart_adb.gold.processing_control'))
    